# Phase 4: Hyperparameter Tuning

In Phase 3 every model used its library's default settings. Here we look for better settings ("hyperparameters") and check how much accuracy that buys, and how much extra compute it costs. Tuning time is itself an energy cost, so it matters for this paper.

| Model | What we tune | Settings tried |
|---|---|---|
| Linear Regression | nothing to tune | – |
| Decision Tree | `max_depth`, `min_samples_split` | 12 |
| Random Forest | `n_estimators`, `max_depth` | 5 (reduced grid, see below) |
| XGBoost | `n_estimators`, `max_depth`, `learning_rate` | 12 |
| LightGBM | `n_estimators`, `max_depth`, `learning_rate` | 12 |
| LSTM | hidden units, history length | 4 |

The exact values are in `scripts/model_definitions.py` (`PARAM_GRIDS`, `LSTM_TUNING_GRID`).

**How the tree models are tuned: GridSearchCV with TimeSeriesSplit (3 folds).** Every combination of settings is tried. To score a combination we can't use the test set (that would be cheating), so the *training* data is split by time into 3 folds:

| Fold | Trains on | Validates on |
|---|---|---|
| 1 | first ¼ of the training period | second ¼ |
| 2 | first ½ | third ¼ |
| 3 | first ¾ | last ¼ |

The setting with the lowest average validation RMSE wins. The model is then retrained with it on the full training set, and only then scored once on the test set.

*Detail that matters:* our training table is stored VM by VM. For the folds to really be periods of time (past → future), we sort the training rows **by timestamp** before splitting. Without that, the "folds" would just be different groups of VMs.

**Why the Random Forest grid is smaller:** on free Colab one default Random Forest takes about 10 minutes. Full-depth trees are therefore only tried with 50 trees: {50 trees: depth 10, 20, unlimited} + {100 trees: depth 10, 20} = 5 settings. The team agreed this on 4 Oct 2026.

**How the LSTM is tuned:** a full grid with cross-validation would take hours on free Colab, so we use a simpler validation-based search instead. Each of the 4 settings (32 or 64 units × 6 or 12 readings of history) is trained once with early stopping. The one with the lowest error on the validation slice (last 10% of each VM's training period) wins, and is then scored on the test set.

**Run time on free Colab:** about 45–60 minutes, mostly Random Forest. **If Colab disconnects, run everything again.** Each model's search is saved when it finishes, and finished models are skipped. Set `FORCE_RERUN = True` to start over.

## 0. Setup

In [1]:
# Mount Google Drive (Colab) or use the local Drive folder (PC), then load our shared code.
import os, sys, importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Green_ML_Project'
except ImportError:
    PROJECT_ROOT = r'H:\My Drive\Green_ML_Project'   # running on the PC instead of Colab

os.environ['GREEN_ML_ROOT'] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts'))

import config as C
import utils as U
importlib.reload(C); importlib.reload(U)   # picks up any edits to the scripts without restarting

print('Project folder:', C.PROJECT_ROOT)
print('Folder found:  ', os.path.isdir(C.PROJECT_ROOT))

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Green_ML_Project
Folder found:   True


In [2]:
import gc, json, os
import numpy as np
import pandas as pd
import joblib, torch
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, ParameterGrid
import model_definitions as M
importlib.reload(M)

FORCE_RERUN = False

pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 80)
log = U.get_logger('phase04_hyperparameter_tuning')
log.info(f'CPU cores: {os.cpu_count()} | CV: TimeSeriesSplit({C.CV_SPLITS}) | scoring: RMSE | FORCE_RERUN = {FORCE_RERUN}')
for name, grid in M.PARAM_GRIDS.items():
    log.info(f'{name}: {len(ParameterGrid(grid))} settings -> {grid}')
log.info(f'LSTM: {len(M.LSTM_TUNING_GRID)} settings -> {M.LSTM_TUNING_GRID}')

RUN STARTED 2026-10-04 05:25:14 | phase04_hyperparameter_tuning
python 3.13.15 | pandas 2.2.3 | numpy 2.1.3 | machine: Linux-6.6.122+-x86_64-with-glibc2.39
project root: /content/drive/MyDrive/Green_ML_Project
CPU cores: 2 | CV: TimeSeriesSplit(3) | scoring: RMSE | FORCE_RERUN = False
Decision Tree: 12 settings -> {'max_depth': [5, 10, 15, None], 'min_samples_split': [2, 10, 50]}
Random Forest: 5 settings -> [{'n_estimators': [50], 'max_depth': [10, 20, None]}, {'n_estimators': [100], 'max_depth': [10, 20]}]
XGBoost: 12 settings -> {'n_estimators': [100, 300], 'max_depth': [4, 6, 8], 'learning_rate': [0.05, 0.1]}
LightGBM: 12 settings -> {'n_estimators': [100, 300], 'max_depth': [4, 8, -1], 'learning_rate': [0.05, 0.1]}
LSTM: 4 settings -> [{'hidden_size': 32, 'seq_len': 6}, {'hidden_size': 32, 'seq_len': 12}, {'hidden_size': 64, 'seq_len': 6}, {'hidden_size': 64, 'seq_len': 12}]


## 1. Load the data and the Phase 3 results

In [3]:
train = pd.read_parquet(C.DATA_PROCESSED / 'train.parquet')
test = pd.read_parquet(C.DATA_PROCESSED / 'test.parquet')
features_main = pd.read_parquet(C.DATA_PROCESSED / 'features_main.parquet')
FEATURES, TARGET = U.FEATURE_COLUMNS, C.TARGET
phase3 = json.load(open(C.MODELS_DIR / 'phase3_results.json'))

# Sort the training rows by time (stable sort keeps VM order within the same timestamp)
train_cv = train.sort_values('timestamp', kind='stable').reset_index(drop=True)
X_cv, y_cv = train_cv[FEATURES], train_cv[TARGET].to_numpy()
X_test, y_test = test[FEATURES], test[TARGET].to_numpy()

tscv = TimeSeriesSplit(n_splits=C.CV_SPLITS)
for k, (a, b) in enumerate(tscv.split(X_cv), 1):
    log.info(f'Fold {k}: train {len(a):,} rows ({train_cv.timestamp.iloc[a[0]]:%m-%d} .. {train_cv.timestamp.iloc[a[-1]]:%m-%d %H:%M}) '
             f'-> validate {len(b):,} rows ({train_cv.timestamp.iloc[b[0]]:%m-%d %H:%M} .. {train_cv.timestamp.iloc[b[-1]]:%m-%d})')

RESULTS_FILE = C.MODELS_DIR / 'phase4_results.json'
results = {} if FORCE_RERUN or not RESULTS_FILE.exists() else json.load(open(RESULTS_FILE))
if results:
    log.info(f'Already finished in an earlier run (will be skipped): {list(results)}')

def save_progress(name, row, predictions):
    results[name] = row
    json.dump(results, open(RESULTS_FILE, 'w'), indent=2)
    np.save(C.MODELS_DIR / f'pred_test_tuned_{M.slug(name)}.npy', predictions)

Fold 1: train 80,281 rows (08-12 .. 08-18 19:41) -> validate 80,278 rows (08-18 19:41 .. 08-24)
Fold 2: train 160,559 rows (08-12 .. 08-24 21:22) -> validate 80,278 rows (08-24 21:22 .. 08-30)
Fold 3: train 240,837 rows (08-12 .. 08-30 22:33) -> validate 80,278 rows (08-30 22:38 .. 09-10)


## 2. Grid search for the four tree models

GridSearchCV prints one line per fit, so you can see it's working. Random Forest is the slow one: 5 settings × 3 folds = 15 fits, plus a final retrain. Expect roughly 30–40 minutes for that cell alone.

We record:
- **tuning time**: the whole search, including the final retrain
- **tuned training time**: just the final retrain with the winning settings. This is the training time of the tuned model, comparable with Phase 3.

In [4]:
for name, grid in M.PARAM_GRIDS.items():
    if name in results:
        log.info(f'{name}: already tuned, skipping')
        continue
    n_settings = len(ParameterGrid(grid))
    log.info(f'{name}: searching {n_settings} settings x {C.CV_SPLITS} folds ...')
    search = GridSearchCV(M.make_model(name, seed=C.RANDOM_SEED), grid, cv=tscv,
                          scoring='neg_root_mean_squared_error', refit=True, n_jobs=1, verbose=2,
                          return_train_score=False)
    with U.Timer() as t_search:
        search.fit(X_cv, y_cv)
    best = search.best_estimator_
    with U.Timer() as t_pred:
        pred = best.predict(X_test)
    scores = U.regression_metrics(y_test, pred)

    cv_table = pd.DataFrame(search.cv_results_)[['params', 'mean_test_score', 'std_test_score', 'mean_fit_time', 'rank_test_score']]
    cv_table['mean_cv_rmse'] = -cv_table.pop('mean_test_score')
    cv_table.sort_values('rank_test_score').to_csv(C.TABLES_DIR / f'phase4_cv_{M.slug(name)}.csv', index=False)
    U.log_df(log, cv_table.sort_values('rank_test_score'), f'{name}: all settings (CV RMSE, lower is better)')

    path = C.MODELS_DIR / f'tuned_{M.slug(name)}.joblib'
    joblib.dump(best, path, compress=3 if name == 'Random Forest' else 0)
    row = {**scores, 'best_params': {k: (None if v is None else v) for k, v in search.best_params_.items()},
           'best_cv_rmse': -search.best_score_, 'n_settings': n_settings,
           'tuning_time_sec': t_search.seconds, 'train_time_sec': search.refit_time_,
           'inference_time_sec': t_pred.seconds, 'model_size_mb': path.stat().st_size / 1e6}
    save_progress(name, row, pred)
    log.info(f'{name}: best {search.best_params_} | CV RMSE {row["best_cv_rmse"]:.4f} | test MAE {scores["MAE"]:.4f} '
             f'RMSE {scores["RMSE"]:.4f} R2 {scores["R2"]:.4f} | tuning {t_search.seconds / 60:.1f} min | '
             f'refit {search.refit_time_:.1f}s')
    del search, best; gc.collect()

Decision Tree: searching 12 settings x 3 folds ...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END ...................max_depth=5, min_samples_split=2; total time=   0.5s
[CV] END ...................max_depth=5, min_samples_split=2; total time=   1.0s
[CV] END ...................max_depth=5, min_samples_split=2; total time=   1.6s
[CV] END ..................max_depth=5, min_samples_split=10; total time=   0.5s
[CV] END ..................max_depth=5, min_samples_split=10; total time=   1.0s
[CV] END ..................max_depth=5, min_samples_split=10; total time=   1.6s
[CV] END ..................max_depth=5, min_samples_split=50; total time=   0.5s
[CV] END ..................max_depth=5, min_samples_split=50; total time=   1.0s
[CV] END ..................max_depth=5, min_samples_split=50; total time=   1.6s
[CV] END ..................max_depth=10, min_samples_split=2; total time=   1.4s
[CV] END ..................max_depth=10, min_samples_split=2; total time=   2.

## 3. LSTM: validation-based search

Each setting is trained once, with early stopping on the validation slice. The winner is the one with the lowest validation error, and **only the winner** is scored on the test set. Each setting's result is saved as soon as it finishes, so a disconnect loses at most one setting.

In [5]:
LSTM_RUNS_FILE = C.MODELS_DIR / 'phase4_lstm_runs.json'
lstm_runs = {} if FORCE_RERUN or not LSTM_RUNS_FILE.exists() else json.load(open(LSTM_RUNS_FILE))

if 'LSTM' in results:
    log.info('LSTM: already tuned, skipping')
else:
    for cfg in M.LSTM_TUNING_GRID:
        key = f"h{cfg['hidden_size']}_s{cfg['seq_len']}"
        if key in lstm_runs:
            log.info(f'LSTM {key}: already done, skipping')
            continue
        ds = M.lstm_datasets(features_main, cfg['seq_len'])
        (Xtr, ytr), (Xva, yva) = ds['train'], ds['val']
        log.info(f'LSTM {key}: hidden={cfg["hidden_size"]}, history={cfg["seq_len"]} readings, '
                 f'{len(Xtr):,} training windows')
        with U.Timer() as t_fit:
            model, history = M.train_lstm(Xtr, ytr, Xva, yva, seed=C.RANDOM_SEED, log_fn=log.info, **cfg)
        val_rmse = float(np.sqrt(min(h['val_loss'] for h in history))) * 100   # back to CPU % points
        torch.save(model.state_dict(), C.MODELS_DIR / f'lstm_search_{key}.pt')
        lstm_runs[key] = {**cfg, 'val_rmse': val_rmse, 'train_time_sec': t_fit.seconds, 'epochs_run': len(history)}
        json.dump(lstm_runs, open(LSTM_RUNS_FILE, 'w'), indent=2)
        log.info(f'LSTM {key}: validation RMSE {val_rmse:.4f} | {t_fit.seconds:.1f}s, {len(history)} epochs')
        del model; gc.collect()

    runs = pd.DataFrame.from_dict(lstm_runs, orient='index').sort_values('val_rmse')
    U.log_df(log, runs, 'LSTM: all settings (validation RMSE, lower is better)')
    runs.to_csv(C.TABLES_DIR / 'phase4_cv_lstm.csv')
    best_key = runs.index[0]
    best_cfg = {'hidden_size': int(runs.loc[best_key, 'hidden_size']), 'seq_len': int(runs.loc[best_key, 'seq_len'])}

    # Score only the winner on the test set
    model = M.build_lstm(len(M.LSTM_SEQ_FEATURES), best_cfg['hidden_size'], M.LSTM_DEFAULTS['num_layers'], C.RANDOM_SEED)
    model.load_state_dict(torch.load(C.MODELS_DIR / f'lstm_search_{best_key}.pt'))
    Xte, yte, _ = M.lstm_datasets(features_main, best_cfg['seq_len'])['test']
    assert len(Xte) == len(test) and np.allclose(yte * 100, y_test, atol=1e-3)
    with U.Timer() as t_pred:
        pred = M.predict_lstm(model, Xte)
    scores = U.regression_metrics(y_test, pred)
    torch.save(model.state_dict(), C.MODELS_DIR / 'tuned_lstm.pt')
    row = {**scores, 'best_params': best_cfg, 'best_cv_rmse': float(runs.loc[best_key, 'val_rmse']),
           'n_settings': len(runs), 'tuning_time_sec': float(runs['train_time_sec'].sum()),
           'train_time_sec': float(runs.loc[best_key, 'train_time_sec']), 'inference_time_sec': t_pred.seconds,
           'model_size_mb': (C.MODELS_DIR / 'tuned_lstm.pt').stat().st_size / 1e6}
    save_progress('LSTM', row, pred)
    log.info(f'LSTM: best {best_cfg} | test MAE {scores["MAE"]:.4f} RMSE {scores["RMSE"]:.4f} R2 {scores["R2"]:.4f} '
             f'| tuning {row["tuning_time_sec"] / 60:.1f} min')

LSTM h32_s6: hidden=32, history=6 readings, 288,736 training windows
  epoch  1: train_mse=0.00372 val_mse=0.00087 (7.8s)
  epoch  2: train_mse=0.00169 val_mse=0.00080 (10.5s)
  epoch  3: train_mse=0.00163 val_mse=0.00080 (6.6s)
  epoch  4: train_mse=0.00162 val_mse=0.00085 (10.4s)
  epoch  5: train_mse=0.00160 val_mse=0.00079 (6.9s)
  epoch  6: train_mse=0.00159 val_mse=0.00083 (10.0s)
  epoch  7: train_mse=0.00158 val_mse=0.00081 (10.3s)
  epoch  8: train_mse=0.00156 val_mse=0.00079 (6.4s)
  early stop at epoch 8 (best val_mse=0.00079)
LSTM h32_s6: validation RMSE 2.8066 | 76.3s, 8 epochs
LSTM h32_s12: hidden=32, history=12 readings, 288,436 training windows
  epoch  1: train_mse=0.00370 val_mse=0.00100 (14.4s)
  epoch  2: train_mse=0.00167 val_mse=0.00080 (14.4s)
  epoch  3: train_mse=0.00161 val_mse=0.00081 (14.4s)
  epoch  4: train_mse=0.00159 val_mse=0.00081 (14.3s)
  epoch  5: train_mse=0.00158 val_mse=0.00078 (14.3s)
  epoch  6: train_mse=0.00157 val_mse=0.00080 (14.4s)
  epoch

## 4. Default vs tuned: did tuning help, and what did it cost?

Linear Regression has no settings to tune, so its "tuned" row is the same as its default row.

The **"Δ MAE %"** column shows how much tuning changed MAE on the test set. Negative means tuning helped.

In [6]:
rows = []
for name in M.TABULAR_MODELS + ['LSTM']:
    d = phase3[name]
    t = results.get(name, {**d, 'best_params': 'none (no hyperparameters)', 'tuning_time_sec': 0.0, 'best_cv_rmse': np.nan})
    rows.append({'Model': name,
                 'Default MAE': d['MAE'], 'Tuned MAE': t['MAE'],
                 'Default RMSE': d['RMSE'], 'Tuned RMSE': t['RMSE'],
                 'Default R2': d['R2'], 'Tuned R2': t['R2'],
                 'Δ MAE %': 100 * (t['MAE'] - d['MAE']) / d['MAE'],
                 'Default train (s)': d['train_time_sec'], 'Tuned train (s)': t['train_time_sec'],
                 'Tuning time (min)': t['tuning_time_sec'] / 60,
                 'Best settings': json.dumps(t['best_params']) if isinstance(t['best_params'], dict) else t['best_params']})
compare = pd.DataFrame(rows)
U.log_df(log, compare.round(4), 'DEFAULT vs TUNED (test set)')
U.save_table(compare.drop(columns='Best settings'), 'phase4_tuning_before_after',
             'Default vs tuned models (test set)', float_fmt='{:.4f}')
compare[['Model', 'Best settings']].to_csv(C.TABLES_DIR / 'phase4_best_settings.csv', index=False)
U.save_table(compare[['Model', 'Best settings']], 'phase4_best_settings', 'Best settings found by tuning')
log.info(f'Total tuning time: {compare["Tuning time (min)"].sum():.1f} min')
compare.round(4)

--- DEFAULT vs TUNED (test set) ---
               Model  Default MAE  Tuned MAE  Default RMSE  Tuned RMSE  Default R2  Tuned R2  Δ MAE %  Default train (s)  Tuned train (s)  Tuning time (min)                                                  Best settings
0  Linear Regression       0.9032     0.9032        4.5899      4.5899      0.9062    0.9062   0.0000             0.3352           0.3352             0.0000                                      none (no hyperparameters)
1      Decision Tree       0.8468     1.0502        5.2473      4.3561      0.8775    0.9156  24.0242            11.4586           2.9743             1.9442                      {"max_depth": 5, "min_samples_split": 10}
2      Random Forest       0.6903     0.6966        3.6853      3.7493      0.9396    0.9374   0.9035           611.5020         233.0275            37.0259                         {"max_depth": 10, "n_estimators": 100}
3            XGBoost       0.7007     0.6974        3.7376      3.7151      0.9378  

,Model,Default MAE,Tuned MAE,Default RMSE,Tuned RMSE,Default R2,Tuned R2,Δ MAE %,Default train (s),Tuned train (s),Tuning time (min),Best settings
0,Linear Regression,0.9032,0.9032,4.5899,4.5899,0.9062,0.9062,0.0000,0.3352,0.3352,0.0000,none (no hyperparameters)
1,Decision Tree,0.8468,1.0502,5.2473,4.3561,0.8775,0.9156,24.0242,11.4586,2.9743,1.9442,"{""max_depth"": 5, ""min_samples_split"": 10}"
2,Random Forest,0.6903,0.6966,3.6853,3.7493,0.9396,0.9374,0.9035,611.5020,233.0275,37.0259,"{""max_depth"": 10, ""n_estimators"": 100}"
3,XGBoost,0.7007,0.6974,3.7376,3.7151,0.9378,0.9386,-0.4693,3.1828,5.6702,2.7904,"{""learning_rate"": 0.1, ""max_depth"": 4, ""n_estimators"": 300}"
4,LightGBM,0.6743,0.6640,3.7123,3.6507,0.9387,0.9407,-1.5223,2.3208,9.1493,2.3874,"{""learning_rate"": 0.05, ""max_depth"": -1, ""n_estimators"": 300}"
5,LSTM,0.8774,0.9238,4.5345,4.4939,0.9085,0.9101,5.2937,217.5143,157.5926,9.2487,"{""hidden_size"": 32, ""seq_len"": 12}"


In [7]:
# Full metrics table for the tuned models (same layout as Phase 3)
tuned = pd.DataFrame([{'Model': n, **{k: (results[n] if n in results else phase3[n])[k] for k in
                      ['MAE', 'RMSE', 'MAPE', 'R2', 'train_time_sec', 'inference_time_sec', 'model_size_mb']}}
                      for n in M.TABULAR_MODELS + ['LSTM']])
tuned.to_csv(C.TABLES_DIR / 'phase4_tuned_results_raw.csv', index=False)
U.save_table(tuned.rename(columns={'MAPE': 'MAPE (%)', 'train_time_sec': 'Train time (s)',
                                   'inference_time_sec': 'Inference time (s)', 'model_size_mb': 'Size (MB)'}),
             'phase4_tuned_results', 'Tuned models on the test set')

# Linear Regression is unchanged: copy its Phase 3 predictions so every model has a "tuned" prediction file
np.save(C.MODELS_DIR / 'pred_test_tuned_linear_regression.npy', np.load(C.MODELS_DIR / 'pred_test_linear_regression.npy'))
log.info('PHASE 4 COMPLETE')
tuned.round(4)

PHASE 4 COMPLETE


,Model,MAE,RMSE,MAPE,R2,train_time_sec,inference_time_sec,model_size_mb
0,Linear Regression,0.9032,4.5899,18.2409,0.9062,0.3352,0.0230,0.0013
1,Decision Tree,1.0502,4.3561,28.0678,0.9156,2.9743,0.0090,0.0061
2,Random Forest,0.6966,3.7493,14.0398,0.9374,233.0275,0.2780,2.9519
3,XGBoost,0.6974,3.7151,13.4264,0.9386,5.6702,0.3209,0.5198
4,LightGBM,0.6640,3.6507,12.4170,0.9407,9.1493,0.5570,0.8425
5,LSTM,0.9238,4.4939,19.7413,0.9101,157.5926,1.0948,0.0216


## Done: what to check

- **Δ MAE %**: did tuning help, and by how much? Small gains for a lot of tuning time are a useful finding in their own right, since the paper is about whether compute is worth spending.
- **Tuning time** vs **default training time**: tuning repeats training many times. That cost is reported separately and isn't part of the TOPSIS training-time criterion.
- **Best settings**: a winner at the edge of a grid (e.g. the largest number of trees) suggests a bigger grid could help a little more. That goes in the Limitations section.

Next: `05_sustainability_tracking.ipynb` measures energy and CO₂.